# Árvore de decisão seguindo as aulas

Construímos uma árvore do zero com a escolha de atributos pelo ganho de informação, como no ID3 apresentado na aula 5, páginas 20 a 28. Para usar os valores numéricos como categorias, agrupamos cada medida em faixas. A discretização aparece na aula 2, páginas 86 a 90.

Comparamos a árvore com o KNN nas mesmas divisões. Depois verificamos se os dois costumam errar as mesmas amostras. Isso nos ajuda a discutir uma possível combinação a partir dos resultados.

As páginas citadas são contadas a partir da capa do PDF. Usamos a biblioteca padrão do Python.

### Base e divisão das amostras

Lemos a base padronizada e repetimos a separação por classe com semente 42. Temos 416 amostras para escolher a configuração e 104 reservadas para comparação local. Essas 104 já foram consultadas nos experimentos anteriores.

O método holdout aparece na aula 7, página 7. A proporção de 80% e 20% e a semente são escolhas deste trabalho. Conferimos também que cada paciente aparece uma única vez.

In [1]:
from pathlib import Path
from collections import Counter
from statistics import median, mean
from itertools import product
import csv
import math
import random

raiz = Path.cwd()
if not (raiz / "dados" / "treinamento_padronizado.csv").is_file():
    raiz = raiz.parent

with (raiz / "dados" / "treinamento_padronizado.csv").open(
    encoding="utf-8-sig", newline=""
) as arquivo:
    amostras = list(csv.DictReader(arquivo))

pacientes = [a["PATIENT_CODE"].strip() for a in amostras]
if any(not p for p in pacientes) or len(set(pacientes)) != len(pacientes):
    raise ValueError("Conferir os registros de paciente antes da divisão.")
if any(a["class"] not in {"0", "1"} for a in amostras):
    raise ValueError("Esperávamos as classes 0 e 1.")

controle = [a for a in amostras if a["class"] == "0"]
diabetes = [a for a in amostras if a["class"] == "1"]
gerador = random.Random(42)
gerador.shuffle(controle)
gerador.shuffle(diabetes)

n0 = round(len(controle) * 0.20)
n1 = round(len(diabetes) * 0.20)
validacao = controle[:n0] + diabetes[:n1]
treinamento = controle[n0:] + diabetes[n1:]

print("Treinamento:", len(treinamento), dict(Counter(a["class"] for a in treinamento)))
print("Validação:", len(validacao), dict(Counter(a["class"] for a in validacao)))

Treinamento: 416 {'0': 200, '1': 216}
Validação: 104 {'0': 50, '1': 54}


### Atributos que vamos comparar

Testamos dois grupos de medidas numéricas. Um contém glicose, hemoglobina e triglicerídeos, que foram escolhidos no KNN. O outro contém as nove medidas originais. Cada grupo é testado sozinho e junto com seis atributos categóricos.

As categorias já foram padronizadas na exploração. Para a árvore, um valor categórico desconhecido recebe a categoria desconhecido. As medidas ausentes são preenchidas pela mediana do treinamento. Mantemos a leitura numérica anterior, incluindo HEMOGLOBIN acima de 100 como ausente sob a premissa de HbA1c.

Seleção de atributos e tratamento de ausências são abordados na aula 2. Essas combinações e as estimativas específicas são nossas escolhas para a base.

In [2]:
colunas_numericas = [
    "AGE", "HEIGHT", "WEIGHT", "SYSTOLIC", "DIASTOLIC",
    "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM",
]
colunas_categoricas = [
    "SMOKER", "FAMILY_HISTORY", "PHYSICAL_ACTIVITY",
    "GENDER", "BLOOD_GROUP", "RH_FACTOR",
]
grupos = {
    "Três exames": ["GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"],
    "Três exames e categorias": ["GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"] + colunas_categoricas,
    "Nove medidas": colunas_numericas,
    "Nove medidas e categorias": colunas_numericas + colunas_categoricas,
}


def ler_numero(amostra, coluna):
    try:
        valor = float(amostra[coluna].strip().replace(",", "."))
    except ValueError:
        return None
    if not math.isfinite(valor) or valor <= 0:
        return None
    if coluna == "HEMOGLOBIN" and valor > 100:
        return None
    return valor


def ler_categoria(amostra, coluna):
    valor = amostra[coluna].strip()
    if valor in {"", "desconhecido"} or (coluna == "GENDER" and valor == "N"):
        return "desconhecido"
    return valor

### Discretização em faixas

O ID3 do exemplo escolhe um atributo e cria um ramo para cada categoria. Para aplicar esse procedimento às medidas, dividimos a distância entre o mínimo e o máximo do treinamento em faixas de mesma largura.

Por exemplo, se o mínimo for 50 e o máximo for 150, duas faixas ficam separadas em 100. Um valor abaixo de 100 entra na primeira faixa e um valor a partir de 100 entra na segunda. O máximo entra na última faixa. Valores de avaliação fora dos extremos entram na primeira ou na última faixa.

A discretização é apresentada na aula 2. Usar faixas de mesma largura e comparar 2, 3, 5 e 10 faixas são decisões de implementação deste experimento. Os limites são calculados novamente com o treinamento de cada rodada.

In [3]:
def calcular_parametros(referencia):
    parametros = {}
    for coluna in colunas_numericas:
        valores = [ler_numero(a, coluna) for a in referencia]
        disponiveis = [v for v in valores if v is not None]
        if not disponiveis:
            raise ValueError(f"Nenhum valor disponível no treinamento para {coluna}.")
        parametros[coluna] = {
            "mediana": median(disponiveis),
            "minimo": min(disponiveis),
            "maximo": max(disponiveis),
        }
    return parametros


def preparar_amostra(amostra, atributos, parametros, faixas):
    resultado = {}
    for atributo in atributos:
        if atributo in colunas_categoricas:
            resultado[atributo] = ler_categoria(amostra, atributo)
            continue

        valor = ler_numero(amostra, atributo)
        p = parametros[atributo]
        if valor is None:
            valor = p["mediana"]

        amplitude = p["maximo"] - p["minimo"]
        if amplitude == 0:
            indice_faixa = 0
        else:
            indice_faixa = math.floor(faixas * (valor - p["minimo"]) / amplitude)
            indice_faixa = max(0, min(faixas - 1, indice_faixa))
        resultado[atributo] = f"faixa_{indice_faixa + 1}"

    return resultado

### Entropia e ganho de informação

A entropia mede o quanto as classes estão misturadas em um grupo. Se todas as amostras forem da mesma classe, ela vale zero. Se metade for controle e metade diabetes, ela vale 1.

Para cada atributo, separamos as amostras pelas categorias e calculamos a entropia de cada grupo. Tiramos uma média dessas entropias, dando mais peso aos grupos maiores. O ganho de informação é a entropia antes da divisão menos essa média. Escolhemos o atributo com maior ganho, como na aula 5, página 27.

In [4]:
def entropia(classes):
    contagem = Counter(classes)
    resultado = 0.0
    for quantidade in contagem.values():
        proporcao = quantidade / len(classes)
        resultado -= proporcao * math.log2(proporcao)
    return resultado


def separar_por_atributo(exemplos, atributo):
    grupos_atributo = {}
    for entrada, classe in exemplos:
        valor = entrada[atributo]
        grupos_atributo.setdefault(valor, []).append((entrada, classe))
    return grupos_atributo


def ganho_informacao(exemplos, atributo):
    antes = entropia([classe for entrada, classe in exemplos])
    depois = 0.0
    for grupo in separar_por_atributo(exemplos, atributo).values():
        peso = len(grupo) / len(exemplos)
        depois += peso * entropia([classe for entrada, classe in grupo])
    return antes - depois


print("Entropia de um grupo com uma só classe:", entropia(["0", "0", "0"]))
print("Entropia de um grupo dividido igualmente:", entropia(["0", "0", "1", "1"]))

Entropia de um grupo com uma só classe: 0.0
Entropia de um grupo dividido igualmente: 1.0


### Construção da árvore

Em cada nó, escolhemos o atributo de maior ganho. Cada categoria desse atributo recebe um ramo. Repetimos o processo dentro de cada ramo, usando os atributos restantes.

Se todas as amostras de um nó tiverem a mesma classe, criamos uma folha com essa classe. Se os atributos acabarem, usamos a classe mais frequente do grupo. Também guardamos a classe mais frequente de cada nó para responder quando uma categoria nova aparecer na previsão. Essas regras seguem a descrição do ID3 na aula 5, páginas 20 a 22.

Para tornar os empates reproduzíveis, usamos a ordem declarada dos atributos nos empates de ganho. Se as duas classes empatarem na quantidade, usamos a classe padrão do nó pai. Na raiz, o padrão é controle. Essas regras de empate são escolhas desta implementação.

In [5]:
def classe_majoritaria(classes, padrao="0"):
    contagem = Counter(classes)
    if contagem["0"] == contagem["1"]:
        return padrao
    return "1" if contagem["1"] > contagem["0"] else "0"


def construir_arvore(exemplos, atributos, padrao="0"):
    classes = [classe for entrada, classe in exemplos]
    maioria = classe_majoritaria(classes, padrao)
    no = {"classe": maioria, "amostras": len(exemplos)}

    if len(set(classes)) <= 1 or not atributos:
        return no

    melhor = max(atributos, key=lambda atributo: ganho_informacao(exemplos, atributo))
    restantes = [atributo for atributo in atributos if atributo != melhor]
    no["atributo"] = melhor
    no["ganho"] = ganho_informacao(exemplos, melhor)
    no["ramos"] = {}

    for valor, grupo in separar_por_atributo(exemplos, melhor).items():
        no["ramos"][valor] = construir_arvore(grupo, restantes, maioria)
    return no


def prever_arvore(entrada, arvore):
    no = arvore
    while "atributo" in no:
        valor = entrada[no["atributo"]]
        if valor not in no["ramos"]:
            return no["classe"]
        no = no["ramos"][valor]
    return no["classe"]

### Métricas e cinco divisões

Usamos as mesmas cinco partes do KNN, com semente 42. Para cada combinação de atributos e quantidade de faixas, calculamos as métricas nas cinco rodadas. Escolhemos pelo maior F1 macro médio.

As métricas são as mesmas do KNN, com controle como classe 0 e diabetes como classe 1. Registramos a matriz na ordem 0 e 1, com as classes reais nas linhas e as previsões nas colunas.

In [6]:
def calcular_metricas(reais, previstas):
    pares = Counter(zip(reais, previstas, strict=True))
    resultado = {
        "acuracia": sum(real == prevista for real, prevista in zip(reais, previstas)) / len(reais)
    }
    f1s = []
    for classe in ["0", "1"]:
        acertos = pares[classe, classe]
        total_real = sum(pares[classe, prevista] for prevista in ["0", "1"])
        total_previsto = sum(pares[real, classe] for real in ["0", "1"])
        precisao = acertos / total_previsto if total_previsto else 0.0
        recall = acertos / total_real if total_real else 0.0
        f1 = 2 * precisao * recall / (precisao + recall) if precisao + recall else 0.0
        resultado[f"precisao_{classe}"] = precisao
        resultado[f"recall_{classe}"] = recall
        f1s.append(f1)
    resultado["precisao_macro"] = (resultado["precisao_0"] + resultado["precisao_1"]) / 2
    resultado["f1_macro"] = mean(f1s)
    resultado["matriz"] = [[pares[real, prevista] for prevista in ["0", "1"]] for real in ["0", "1"]]
    return resultado


partes = [[] for _ in range(5)]
gerador_partes = random.Random(42)
for classe in ["0", "1"]:
    grupo = [a for a in treinamento if a["class"] == classe]
    gerador_partes.shuffle(grupo)
    for indice, amostra in enumerate(grupo):
        partes[indice % 5].append(amostra)

configuracoes = list(product(grupos, [2, 3, 5, 10]))
resultados = {config: [] for config in configuracoes}

for indice, avaliacao in enumerate(partes):
    referencia = [a for j, parte in enumerate(partes) if j != indice for a in parte]
    parametros = calcular_parametros(referencia)
    reais = [a["class"] for a in avaliacao]

    for nome, faixas in configuracoes:
        atributos = grupos[nome]
        exemplos = [
            (preparar_amostra(a, atributos, parametros, faixas), a["class"])
            for a in referencia
        ]
        arvore = construir_arvore(exemplos, atributos)
        previstas = [
            prever_arvore(preparar_amostra(a, atributos, parametros, faixas), arvore)
            for a in avaliacao
        ]
        resultados[nome, faixas].append(calcular_metricas(reais, previstas))

In [7]:
print("Atributos                   | Faixas | Acurácia | F1 macro | Recall 0 | Recall 1")
for (nome, faixas), rodadas in resultados.items():
    print(
        f"{nome:27} | {faixas:>6} | "
        f"{mean(r['acuracia'] for r in rodadas):>8.2%} | "
        f"{mean(r['f1_macro'] for r in rodadas):>8.2%} | "
        f"{mean(r['recall_0'] for r in rodadas):>8.2%} | "
        f"{mean(r['recall_1'] for r in rodadas):>8.2%}"
    )

melhor_configuracao = min(
    configuracoes,
    key=lambda config: (
        -mean(r["f1_macro"] for r in resultados[config]),
        len(grupos[config[0]]), config[1], configuracoes.index(config),
    )
)
melhor_grupo, melhores_faixas = melhor_configuracao
print()
print("Grupo escolhido:", melhor_grupo)
print("Quantidade de faixas:", melhores_faixas)

Atributos                   | Faixas | Acurácia | F1 macro | Recall 0 | Recall 1
Três exames                 |      2 |   55.52% |   46.60% |  100.00% |   14.33%
Três exames                 |      3 |   64.18% |   59.78% |  100.00% |   31.01%
Três exames                 |      5 |   68.99% |   67.70% |   74.00% |   64.33%
Três exames                 |     10 |   73.32% |   73.05% |   73.50% |   73.14%
Três exames e categorias    |      2 |   55.28% |   55.13% |   54.00% |   56.47%
Três exames e categorias    |      3 |   59.84% |   59.58% |   56.00% |   63.36%
Três exames e categorias    |      5 |   62.25% |   61.97% |   58.00% |   66.22%
Três exames e categorias    |     10 |   67.30% |   67.18% |   65.50% |   69.01%
Nove medidas                |      2 |   66.09% |   65.69% |   64.00% |   68.05%
Nove medidas                |      3 |   64.42% |   64.17% |   60.50% |   68.03%
Nove medidas                |      5 |   66.34% |   66.16% |   63.50% |   69.01%
Nove medidas                

### Árvore escolhida e avaliação local

Ajustamos a discretização e construímos a árvore com as 416 amostras. Depois fazemos previsões para as 104 reservadas.

A tabela dos parâmetros permite recuperar o significado das faixas. A largura de cada faixa é a amplitude do atributo dividida pela quantidade de faixas. O código mostra também os primeiros testes da árvore.

In [8]:
atributos_finais = grupos[melhor_grupo]
parametros_finais = calcular_parametros(treinamento)
exemplos_finais = [
    (preparar_amostra(a, atributos_finais, parametros_finais, melhores_faixas), a["class"])
    for a in treinamento
]
arvore_final = construir_arvore(exemplos_finais, atributos_finais)
reais_validacao = [a["class"] for a in validacao]
previstas_arvore = [
    prever_arvore(preparar_amostra(a, atributos_finais, parametros_finais, melhores_faixas), arvore_final)
    for a in validacao
]
metricas_arvore = calcular_metricas(reais_validacao, previstas_arvore)

print("Atributo         | Mínimo | Máximo | Largura da faixa")
for atributo in atributos_finais:
    if atributo in parametros_finais:
        p = parametros_finais[atributo]
        largura = (p["maximo"] - p["minimo"]) / melhores_faixas
        print(f"{atributo:16} | {p['minimo']:>6.2f} | {p['maximo']:>6.2f} | {largura:>16.2f}")


def mostrar_arvore(no, profundidade=0):
    recuo = "  " * profundidade
    if "atributo" not in no:
        print(f"{recuo}Classe {no['classe']} | {no['amostras']} amostras")
        return
    print(
        f"{recuo}Testar {no['atributo']} | Ganho {no['ganho']:.3f} | "
        f"{no['amostras']} amostras"
    )
    if profundidade >= 2:
        return
    for valor, filho in sorted(no["ramos"].items()):
        print(f"{recuo}Se {no['atributo']} = {valor}")
        mostrar_arvore(filho, profundidade + 1)


print()
print("Primeiros níveis da árvore:")
mostrar_arvore(arvore_final)

Atributo         | Mínimo | Máximo | Largura da faixa
GLUCOSE          |   3.10 | 500.00 |            49.69
HEMOGLOBIN       |   4.70 |  88.00 |             8.33
TRIGLYCERIDES    |  21.00 | 2880.00 |           285.90

Primeiros níveis da árvore:
Testar GLUCOSE | Ganho 0.234 | 416 amostras
Se GLUCOSE = faixa_1
  Testar HEMOGLOBIN | Ganho 0.090 | 37 amostras
  Se HEMOGLOBIN = faixa_1
    Testar TRIGLYCERIDES | Ganho 0.000 | 32 amostras
  Se HEMOGLOBIN = faixa_4
    Testar TRIGLYCERIDES | Ganho 0.000 | 2 amostras
  Se HEMOGLOBIN = faixa_5
    Classe 1 | 1 amostras
  Se HEMOGLOBIN = faixa_8
    Classe 1 | 2 amostras
Se GLUCOSE = faixa_10
  Classe 1 | 1 amostras
Se GLUCOSE = faixa_2
  Testar HEMOGLOBIN | Ganho 0.015 | 143 amostras
  Se HEMOGLOBIN = faixa_1
    Testar TRIGLYCERIDES | Ganho 0.006 | 131 amostras
  Se HEMOGLOBIN = faixa_3
    Classe 0 | 2 amostras
  Se HEMOGLOBIN = faixa_4
    Testar TRIGLYCERIDES | Ganho 0.000 | 8 amostras
  Se HEMOGLOBIN = faixa_5
    Classe 0 | 2 amostras
Se

### Comparação com o KNN

Usamos o KNN escolhido no notebook de seleção de atributos, com glicose, hemoglobina e triglicerídeos, mínimo e máximo, Manhattan e K igual a 9. Reproduzimos as previsões usando as mesmas 416 amostras como referência e as mesmas 104 para avaliação.

Comparamos cada modelo sozinho. O ganho de combinar os dois depende de como seriam resolvidas as discordâncias, uma regra que ainda precisaríamos definir e testar.

In [9]:
atributos_knn = ["GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"]


def normalizar_knn(amostra):
    resultado = []
    for atributo in atributos_knn:
        p = parametros_finais[atributo]
        valor = ler_numero(amostra, atributo)
        if valor is None:
            valor = p["mediana"]
        amplitude = p["maximo"] - p["minimo"]
        resultado.append((valor - p["minimo"]) / amplitude if amplitude else 0.0)
    return resultado


referencia_knn = [(normalizar_knn(a), a["class"]) for a in treinamento]
random.Random(42).shuffle(referencia_knn)
previstas_knn = []

for amostra in validacao:
    numeros = normalizar_knn(amostra)
    vizinhos = []
    for numeros_ref, classe in referencia_knn:
        distancia = sum(abs(x - y) for x, y in zip(numeros, numeros_ref, strict=True))
        vizinhos.append((distancia, classe))
    vizinhos.sort(key=lambda v: v[0])
    votos = Counter(classe for distancia, classe in vizinhos[:9])
    previstas_knn.append(votos.most_common(1)[0][0])

metricas_knn = calcular_metricas(reais_validacao, previstas_knn)

print("Modelo  | Acertos | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for nome, metricas in [("KNN", metricas_knn), ("Árvore", metricas_arvore)]:
    matriz = metricas["matriz"]
    acertos = matriz[0][0] + matriz[1][1]
    print(
        f"{nome:7} | {acertos:>7} | {metricas['acuracia']:>8.2%} | "
        f"{metricas['f1_macro']:>8.2%} | {metricas['precisao_macro']:>8.2%} | "
        f"{metricas['recall_0']:>8.2%} | {metricas['recall_1']:>8.2%}"
    )
    print("Matriz:", matriz)

complementaridade = Counter()
for real, knn, arvore in zip(reais_validacao, previstas_knn, previstas_arvore, strict=True):
    if knn == real and arvore == real:
        grupo = "Os dois acertaram"
    elif knn == real:
        grupo = "Só o KNN acertou"
    elif arvore == real:
        grupo = "Só a árvore acertou"
    else:
        grupo = "Os dois erraram"
    complementaridade[grupo] += 1

print()
for grupo in ["Os dois acertaram", "Só o KNN acertou", "Só a árvore acertou", "Os dois erraram"]:
    print(f"{grupo}: {complementaridade[grupo]}")

Modelo  | Acertos | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
KNN     |      88 |   84.62% |   84.56% |   84.67% |   82.00% |   87.04%
Matriz: [[41, 9], [7, 47]]
Árvore  |      66 |   63.46% |   63.12% |   63.48% |   56.00% |   70.37%
Matriz: [[28, 22], [16, 38]]

Os dois acertaram: 65
Só o KNN acertou: 23
Só a árvore acertou: 1
Os dois erraram: 15


#### Leitura desta execução

A configuração escolhida foi três exames, com 10 faixas por medida. A árvore teve acurácia de 63.46% e F1 macro de 63.12% nas 104 amostras reservadas. O KNN teve 84.62% de acurácia e 84.56% de F1 macro nessas mesmas amostras.

Os dois acertaram juntos 65 amostras. Só o KNN acertou 23, só a árvore acertou 1, e os dois erraram 15. Ainda precisaríamos de uma regra para decidir nas discordâncias antes de avaliar uma combinação.

### Referências

- [Aula 2](../docs/materiais_aula/aula2_MD.pdf), tratamento de ausências, seleção de atributos e discretização.
- [Aula 4](../docs/materiais_aula/aula4_MD.pdf), página 9, média e mediana.
- [Aula 5](../docs/materiais_aula/aula5_MD.pdf), páginas 16 a 28, árvore de decisão e ID3.
- [Aula 7](../docs/materiais_aula/aula7_MD.pdf), páginas 7 a 11, avaliação e métricas.

O ID3 e o ganho de informação vêm do material. As faixas, os grupos de atributos, os critérios de empate e a mediana para preencher ausências são decisões da implementação.